# Phase 1 — Baselines and the λ sweep

Run `make sample` first (or point `DATA` at `data/raw`).

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from rewear.data import load_raw, temporal_split
from rewear.baseline import PopularityRecommender, ItemKNNRecommender
from rewear.sustainability import build_article_features, SustainabilityReranker
from rewear.metrics import evaluate

DATA = '../data/sample'
ds = load_raw(DATA)
split = temporal_split(ds.transactions, val_days=7)
truth = split.ground_truth()
targets = list(truth)
n_catalog = split.train['article_id'].nunique()
len(split.train), len(split.val), len(targets)

In [ ]:
knn = ItemKNNRecommender().fit(split.train)
knn_recs = knn.recommend(targets, k=60)
feats = build_article_features(split.train, ds.articles)

rows = [evaluate('popularity', PopularityRecommender().fit(split.train).recommend(targets), truth, split.train, n_catalog)]
for lam in [0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    rows.append(evaluate(f'rewear λ={lam}', SustainabilityReranker(feats, lam=lam).rerank(knn_recs), truth, split.train, n_catalog))
res = pd.DataFrame(rows); res

In [ ]:
ax = res.iloc[1:].plot(x='novelty', y='map@12', marker='o', legend=False)
ax.set_title('The trade-off: every step right costs accuracy'); ax.set_xlabel('novelty'); ax.set_ylabel('MAP@12');